# Qwen3.5-9B with Camila Blank's J-lens and R-lens

Run this notebook from top to bottom on a RunPod with one CUDA GPU. A 48 GB GPU is the comfortable minimum. The first run downloads the BF16 model plus about 2.1 GB of lens checkpoints.

In [ ]:
import os
import torch
import transformers
import jlens

assert torch.cuda.is_available(), "No CUDA GPU is visible to this notebook"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
print(f"PyTorch: {torch.__version__}; Transformers: {transformers.__version__}")
print("Hugging Face cache:", os.environ.get("HF_HOME", "~/.cache/huggingface"))

## Load Qwen

Qwen3.5-9B is packaged as a multimodal model. `jlens` automatically selects its text decoder, which is the activation basis Camila's checkpoints target.

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-9B"
LENS_REPO = "camilablank/workspace-lenses"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
hf_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
hf_model.eval()
print("Loaded:", type(hf_model).__name__)
print(f"CUDA allocated: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## Load both lenses

R-lens is a second checkpoint in the same format, not another Python package.

In [ ]:
lens_model = jlens.from_hf(hf_model, tokenizer)
j_lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename="qwen3.5-9b/j-lens/lens.pt"
)
r_lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename="qwen3.5-9b/r-lens/lens.pt"
)

assert lens_model.d_model == j_lens.d_model == r_lens.d_model == 4096
print(lens_model)
print("J-lens:", j_lens)
print("R-lens:", r_lens)

## Smoke test: compare the layer-by-layer readouts

The readout is taken at the final prompt token, so each row shows tokens that layer is poised to produce next.

In [ ]:
PROMPT = "The capital of France is"
LAYERS = [layer for layer in [4, 8, 12, 16, 20, 24, 28, 30] if layer in j_lens.source_layers]

def top_tokens(logits, k=5):
    token_ids = logits[0].topk(k).indices.tolist()
    return [repr(tokenizer.decode([token_id])) for token_id in token_ids]

def run_lens(name, lens):
    lens_logits, model_logits, input_ids = lens.apply(
        lens_model, PROMPT, layers=LAYERS, positions=[-1]
    )
    print(f"\n{name}: {PROMPT!r}")
    for layer in LAYERS:
        print(f"layer {layer:>2}:", top_tokens(lens_logits[layer]))
    print("model   :", top_tokens(model_logits))
    return lens_logits

j_logits = run_lens("J-lens", j_lens)
r_logits = run_lens("R-lens", r_lens)

If those cells finish and print token lists for both lenses, the model, CUDA kernels, checkpoint loading, and Jupyter environment are all working. Change `PROMPT`, `LAYERS`, and `positions` for your experiments.